<a href="https://colab.research.google.com/github/acnoori/EE434/blob/main/SmartGrid1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB 1: ACTIVE DISTRIBUTION FEEDER DYNAMICS & MITIGATION SIMULATION

**Course**: Intelligent Electric Energy Networks (Smart Grids)

**Session 1**: Introduction & Evolution of Smart Grids

**Scenario**: Radial Feeder with 3.5 MW PV + 2.2 MW EV + 1.0 MW BESS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. SYSTEM & FEEDER PARAMETERS

In [ ]:
S_base_mva = 5.0  # MVA Base
V_base_kv = 11.0  # kV Base (Medium Voltage Feeder)
Z_base_ohm = (V_base_kv ** 2) / S_base_mva  # 24.2 Ohms

# Feeder Line Impedance (Realistic MV line: High R/X ratio ~ 2.0)
R_line_ohm = 2.42  # Resistance (Ohms) -> R_pu = 0.10
X_line_ohm = 1.21  # Inductive Reactance (Ohms) -> X_pu = 0.05

R_pu = R_line_ohm / Z_base_ohm
X_pu = X_line_ohm / Z_base_ohm

V_substation = 1.0  # Substation Voltage (p.u.)

## 2. TIME HORIZON & PROFILE GENERATION (24 Hours)

In [ ]:
time_hours = np.linspace(0, 24, 97)  # 15-minute resolution

# Base Residential Load Profile (Peak 1.5 MW at 20:00)
base_load_mw = 0.6 + 0.9 * np.exp(-((time_hours - 20) ** 2) / 16) + 0.3 * np.exp(-((time_hours - 8) ** 2) / 9)

# Solar PV Generation Profile (Peak 3.5 MW at 12:00)
sin_val = np.sin(np.pi * (time_hours - 6) / 12)
pv_gen_mw = np.where((time_hours >= 6) & (time_hours <= 18), 3.5 * (np.maximum(0, sin_val) ** 1.5), 0.0)

# Uncontrolled EV Charging Profile (Peak 2.2 MW between 18:00 - 22:00)
ev_charge_uncontrolled_mw = np.where((time_hours >= 17) & (time_hours <= 23), 2.2 * np.exp(-((time_hours - 20) ** 2) / 6), 0.0)

# Reactive Power Assumptions (Load PF = 0.92 lagging, EV PF = 0.98 lagging)
Q_load_mvar = base_load_mw * np.tan(np.arccos(0.92))
Q_ev_mvar = ev_charge_uncontrolled_mw * np.tan(np.arccos(0.98))

## 3. BASELINE UNCONTROLLED SIMULATION

In [ ]:
# Net Power Injection at feeder node: P_net = P_load + P_EV - P_PV
P_net_uncontrolled_mw = base_load_mw + ev_charge_uncontrolled_mw - pv_gen_mw
Q_net_uncontrolled_mvar = Q_load_mvar + Q_ev_mvar

P_net_uncontrolled_pu = P_net_uncontrolled_mw / S_base_mva
Q_net_uncontrolled_pu = Q_net_uncontrolled_mvar / S_base_mva

# Linearized Feeder Voltage Equation: V_node ≈ V_sub - (R*P_net + X*Q_net) / V_sub
V_node_uncontrolled = V_substation - (R_pu * P_net_uncontrolled_pu + X_pu * Q_net_uncontrolled_pu) / V_substation

## 4. SMART GRID MITIGATION SIMULATION

In [ ]:
# BESS Dispatch: Charge 1.0 MW during midday solar peak, Discharge 1.0 MW during evening EV peak
bess_dispatch_mw = np.zeros_like(time_hours)
bess_dispatch_mw[(time_hours >= 11) & (time_hours <= 14)] = 1.0  # Charging (+ Load)
bess_dispatch_mw[(time_hours >= 18.5) & (time_hours <= 21.5)] = -1.0  # Discharging (- Load)

# Smart Inverter Volt/VAR Control: Inductive Q absorption when V_node > 1.03 p.u.
Q_pv_inverter_mvar = np.zeros_like(time_hours)
for idx in range(len(time_hours)):
    if V_node_uncontrolled[idx] > 1.03:
        Q_pv_inverter_mvar[idx] = 0.8  # 0.8 MVAR inductive absorption

P_net_mitigated_mw = base_load_mw + ev_charge_uncontrolled_mw + bess_dispatch_mw - pv_gen_mw
Q_net_mitigated_mvar = Q_load_mvar + Q_ev_mvar + Q_pv_inverter_mvar

P_net_mitigated_pu = P_net_mitigated_mw / S_base_mva
Q_net_mitigated_pu = Q_net_mitigated_mvar / S_base_mva

V_node_mitigated = V_substation - (R_pu * P_net_mitigated_pu + X_pu * Q_net_mitigated_pu) / V_substation

## 5. RESULTS SUMMARY & PLOTTING

In [ ]:
print(f"--- SIMULATION RESULTS ---")
print(f"Uncontrolled Max Voltage : {np.max(V_node_uncontrolled):.4f} p.u. (Statutory limit: 1.05 p.u.)")
print(f"Uncontrolled Min Voltage : {np.min(V_node_uncontrolled):.4f} p.u. (Statutory limit: 0.95 p.u.)")
print(f"Mitigated Max Voltage    : {np.max(V_node_mitigated):.4f} p.u.")
print(f"Mitigated Min Voltage    : {np.min(V_node_mitigated):.4f} p.u.")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

# Power Profiles Plot
ax1.plot(time_hours, base_load_mw, 'k--', label='Base Residential Load (MW)', linewidth=1.5)
ax1.plot(time_hours, pv_gen_mw, 'g-', label='3.5 MW Solar PV Generation', linewidth=2)
ax1.plot(time_hours, ev_charge_uncontrolled_mw, 'r-', label='2.2 MW Uncontrolled EV Charging', linewidth=2)
ax1.plot(time_hours, bess_dispatch_mw, 'b-.', label='1.0 MW BESS Dispatch (+Charge / -Discharge)', linewidth=1.8)
ax1.axhline(0, color='gray', linestyle=':', alpha=0.6)
ax1.set_ylabel('Power (MW)', fontsize=11, fontweight='bold')
ax1.set_title('Lab 1: Distribution Feeder Active Power Profiles', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper left', fontsize=9)

# Voltage Profile Plot
ax2.plot(time_hours, V_node_uncontrolled, 'r-o', label='Uncontrolled Voltage (Over-voltage & Sag)', linewidth=2, markersize=3)
ax2.plot(time_hours, V_node_mitigated, 'g-s', label='Mitigated Voltage (Smart Inverter Volt/VAR + BESS)', linewidth=2, markersize=3)
ax2.axhline(1.05, color='darkorange', linestyle='--', label='Statutory Upper Voltage Limit (1.05 p.u.)', linewidth=1.5)
ax2.axhline(0.95, color='darkorange', linestyle='--', label='Statutory Lower Voltage Limit (0.95 p.u.)', linewidth=1.5)
ax2.axhline(1.00, color='gray', linestyle=':', alpha=0.6)
ax2.set_xlabel('Time of Day (Hours)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Node Voltage (p.u.)', fontsize=11, fontweight='bold')
ax2.set_xlim(0, 24)
ax2.set_ylim(0.92, 1.08)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()